# 04 Statistical Analysis

Objective: test whether the main categorical and numeric features are meaningfully associated with 30-day readmission.

In [ ]:
from pathlib import Path

import pandas as pd
from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

clean_path = ROOT / 'data' / 'processed' / 'clean_diabetes.csv'
df = pd.read_csv(clean_path)

In [ ]:
categorical_features = [
    'age',
    'gender',
    'race',
    'admission_type',
    'A1Cresult',
    'insulin',
    'prior_inpatient_group',
    'prior_emergency_group',
    'medication_burden_group',
    'stay_length_group',
]

numeric_features = [
    'time_in_hospital',
    'num_medications',
    'number_emergency',
    'number_inpatient',
]

In [ ]:
def run_chi_square(frame, column):
    table = pd.crosstab(frame[column], frame['readmitted_30'])
    chi2, p_value, _, _ = chi2_contingency(table)
    return {'feature': column, 'chi2': chi2, 'p_value': p_value}

chi_square_results = pd.DataFrame([run_chi_square(df, column) for column in categorical_features])
chi_square_results.sort_values('p_value')

In [ ]:
def run_mann_whitney(frame, column):
    readmitted = frame.loc[frame['readmitted_30'] == 1, column]
    not_readmitted = frame.loc[frame['readmitted_30'] == 0, column]
    stat, p_value = mannwhitneyu(readmitted, not_readmitted, alternative='two-sided')
    return {
        'feature': column,
        'readmitted_mean': readmitted.mean(),
        'not_readmitted_mean': not_readmitted.mean(),
        'p_value': p_value,
    }

numeric_results = pd.DataFrame([run_mann_whitney(df, column) for column in numeric_features])
numeric_results.sort_values('p_value')

In [ ]:
X = df[categorical_features + numeric_features]
y = df['readmitted_30']

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('numeric', StandardScaler(), numeric_features),
    ]
)

model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, predictions))
print({'roc_auc': roc_auc_score(y_test, probabilities)})